This notebook showcases a simple way to load data from a Snowflake database into Databricks Delta tables with liquid clustering.

In [0]:
"""
Job Widgets:
- date_cutoff: use if you need to manually replay data from some point forward
- incremental: whether it should be a full or partial load
- source_table: name of the table to load (must match a key in table_json)
"""

In [0]:
%run ./utils_file

Note: I have designed this so that each source schema has its JSON object. This is a choice and you could put everything into one object if desired. Each key in the JSON will represent a table and the values underneath are the config options.

In [0]:
%run ./obs_tables_config_json

In [0]:
from datetime import date, timedelta, datetime
from pyspark.sql.functions import col

In [0]:
#define widgets
dbutils.widgets.text('source_table', 'customer_fact')
dbutils.widgets.text('incremental', 'True')
dbutils.widgets.text('date_cutoff', '')

#get source values
source_table = dbutils.widgets.get('source_table')
snowflake_database = table_json[source_table]['snowflake_database']
snowflake_schema = table_json[source_table]['snowflake_schema']
source_incremental_column = table_json[source_table]['source_incremental_column']

#get target variables
target_catalog = table_json[source_table]['target_catalog']
target_schema = table_json[source_table]['target_schema']
target_table = table_json[source_table]['target_table']
target_merge_columns = table_json[source_table]['target_merge_columns']
target_incremental_column = table_json[source_table]['target_incremental_column']
target = target_catalog.lower() + '.' + target_schema.lower() + '.' + target_table.lower()

#get cluster_by config - defaults to ["auto"] if not provided
cluster_by = table_json[source_table].get('cluster_by', ['auto'])
cluster_by = validate_cluster_by(cluster_by)

#define run variables
date_cutoff = dbutils.widgets.get('date_cutoff')
incremental = dbutils.widgets.get('incremental')

#define merge variable which can be used in the case a schema change has occurred
incremental_load = eval(incremental)

In [0]:
source_df = get_snowflake_data(snowflake_database, snowflake_schema, source_table)

#If this step is taking too long then optimization should be made on the Snowflake side
if len(date_cutoff) != 0:
    #if a date_cutoff is provided, filter from that point forward
    source_df = source_df.filter(col(f'{source_incremental_column}') >= date_cutoff)
elif incremental_load:
    target_df = spark.table(target)
    max_date = target_df.agg({f"{target_incremental_column}": "max"}).collect()[0][0]
    source_df = source_df.filter(col(f'{source_incremental_column}') >= max_date)

In [0]:
source_df.display()

In [0]:
separator = ', '
target_merge_columns_clean = separator.join(target_merge_columns)
match_columns, update_columns, insert_columns = generate_match_insert_columns(target_merge_columns_clean, source_df)

# Generate the dynamic MERGE statement
merge_statement = dynamic_merge_sql(target, match_columns, update_columns, insert_columns)

if incremental_load == False:
    # Full load: write with liquid clustering
    # Build the CLUSTER BY clause
    if cluster_by == ['auto']:
        cluster_clause = "CLUSTER BY AUTO"
    else:
        cluster_cols = ', '.join(cluster_by)
        cluster_clause = f"CLUSTER BY ({cluster_cols})"

    # Write the data to a temp table first, then recreate with clustering
    source_df.createOrReplaceTempView('full_load_data')
    
    # Drop existing table if it exists and recreate with clustering
    spark.sql(f"DROP TABLE IF EXISTS {target}")
    spark.sql(f"""
        CREATE TABLE {target}
        {cluster_clause}
        AS SELECT * FROM full_load_data
    """)
else:
    # Incremental load: merge into existing table
    source_df.createOrReplaceTempView('updates')
    spark.sql(merge_statement)

In [0]:
#as an organization, you should determine what the vacuum period should be and create a common value so that everyone uses the same value here
spark.sql(f"VACUUM {target} RETAIN 160 HOURS")

In [0]:
# Use this also to drop tables and not have to worry about files not being deleted
# spark.conf.set('spark.databricks.delta.retentionDurationCheck.enabled', False)
# spark.sql(f'TRUNCATE TABLE {target}')
# spark.sql(f"VACUUM {target} RETAIN 0 HOURS")
# spark.sql(f"DROP TABLE {target}")